# 2 Baseline: Model Without Context

**Purpose**:
Establish a "before" snapshot so improvements later are visible, attributable, and defensible.
What we're doing

We will ask a small set of domain-specific questions without providing any of the customer documents. This simulates what happens when a customer "just tries" a model before connecting their data.

Why we're doing it

* To create a reference point
* To reveal the gap between fluency and groundedness
* To produce a baseline artifact we can later compare against RAG results

What to look for

* Confident language + incorrect facts
* Vague responses where specificity is required
* Inconsistent terminology across runs

Output

A small baseline dataset:
question → model answer → timestamp → metadata
 saved for later comparison and evaluation.

## 2.1 Setup a Key on MaaS

Before we can interact with any language model, we need an API key from the **Model as a Service (MaaS)** platform. This key authenticates every request we send and tracks usage against your account.


Using your Red Hat SSO log into the Red Hat Demo Platform MaaS service home page:

> https://litellm-prod-frontend.apps.maas.redhatworkshops.io/home

Click on Subscriptions (1) to add or remove the Models that you are subscribed to. The figure below shows an example of an empty subscription list. Click on New Subscription (2) to subscribe to one or more models.

![Empty Subscriptions Page](images/empty_subscription_page.png)


>Note: Subscriptions act as a strategic abstraction layer that turns raw, unvetted AI models into governed, API-driven services, decoupling the complexity of infrastructure and compliance from the developer’s workflow.


You will now see the list of available models. Click on `granite-3-2-8b-instruct` to see the model card.


![Subscribe](images/subscribe.png)

On the model card, click on the Subscribe button. 

![Model Card](images/modelcard.png)

Repeat the process for the following models:

* Granite-4-0-h-tiny
* Microsoft-phi-4
* qwen3-14b

Now click on the API Keys menu item on the left hand side of the page to create an API key.

![Model Card](images/apikeys.png)

Click on the API Keys item and then in the following page click on Create API key.


![Create API Keys](images/createapikeys.png)


In the dialog box that appears, enter a unique name for the Name and, optionally, a description of what the key will be used for.  Next click on the Select All checkbox to add all the models you subscribed to in the previous step to this API key.

![Create API Keys](images/selectall.png)


Click on Create API Key to complete the process and close the dialog. In a moment, you will see your new API Key. Click on the copy button to copy it to your clipboard. Save it for later. Click on close to close out the dialog.

![API Successfully Created](images/success.png)

You will now see your new key in the list. Click on View Key to get more information.


![View Key](images/apikeys_again.png)

The information screen not only lets you retrieve your key, but it also displays the endpoint for the service. Make note of the API URL and save, we will use it in the next step.

![Endpoint Dialog](images/endpoint.png)


> **Tip:** Keep this browser tab open. You will need both the API key and the endpoint URL in the next step.


## 2.1.1 Create your .env file

With your API key in hand, we need to store it — along with the endpoint URL — in a `.env` file at the root of the project directory. This file keeps sensitive configuration out of our notebook code and makes it easy to swap credentials later without modifying any cells.

The `.env` file will contain two values:

| Variable | Description |
|---|---|
| `API_KEY` | The key you just generated on the MaaS platform |
| `ENDPOINT_BASE` | The base URL for the MaaS API (found on the key details screen) |

In the cell below, **uncomment the lines** and replace `APIKEY` with the key you copied and update the endpoint URL if it differs from the default. Then run the cell to write the file.

In [1]:
# !echo 'API_KEY=APIKEY' > ../.env
# !echo 'ENDPOINT_BASE=https://litellm-prod.apps.maas.redhatworkshops.io/v1' >> ../.env

Check the that the data is correct

## 2.2 Testing the connection and API Key

In your favorite text editor, in the root q1labs directory, create a file named .env and enter the following, replacing KEY with the API key provided and ENDPOINT with the URL provided.

```
API_KEY=KEY
ENDPOINT_BASE=ENDPOINT

```
Run the next code cell to will load the appropriate values. 


### 2.2.1 Load Values from .env

In [4]:
import sys
sys.path.insert(0, "..")
from config import API_KEY as key, ENDPOINT_BASE as endpoint_base
import requests
url_models = f"{endpoint_base}/models"
url_chat = f"{endpoint_base}/chat/completions"

print(f"Endpoint: {endpoint_base}")
print(f"API Key:  {key[:8]}...")


Endpoint: https://litellm-prod.apps.maas.redhatworkshops.io/v1
API Key:  sk-lfxgJ...


Notice that the API key is intentionally truncated in the output above. Displaying a full key in a notebook is a security risk: anyone who can see the output (a screenshot, a shared file, a recorded demo) would have a working credential. Showing just the first few characters is enough to confirm the right key loaded without exposing anything sensitive.



### 2.2.2 Test the Connection to MaaS Server`

The following code will test the connection to the MaaS endpoint and ensure you can communicate with the LLMs hosted there.

With the connection working, let’s move on to see what models are available.

In [5]:
headers = {
    "Authorization": f"Bearer {key}",
    "Content-Type": "application/json"
}
data = {
    "model": "granite-3-2-8b-instruct",
    "messages": [{"role": "user", "content": "Hello, world!"}]
}

response = requests.post(url_chat, headers=headers, json=data)
print(response.json())

{'id': 'chatcmpl-681b23990d084fc78e8160eda878adf8', 'created': 1772118593, 'model': 'granite-3-2-8b-instruct', 'object': 'chat.completion', 'system_fingerprint': None, 'choices': [{'finish_reason': 'stop', 'index': 0, 'message': {'content': 'Hello! How can I assist you today?', 'role': 'assistant', 'tool_calls': None, 'function_call': None}, 'provider_specific_fields': {'stop_reason': None}}], 'usage': {'completion_tokens': 10, 'prompt_tokens': 12, 'total_tokens': 22, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'service_tier': None, 'prompt_logprobs': None, 'kv_transfer_params': None}


## 2.3 Listing the Models Available to the API Key

In [11]:
headers = {"Authorization": f"Bearer {key}"}

response = requests.get(url_models, headers=headers)
models = response.json()

print("Models available")

# Loop through and print just the names
for model in models['data']:
    print(f"\t- {model['id']}")

Models available
	- granite-3-2-8b-instruct
	- qwen3-14b
	- microsoft-phi-4
	- granite-4-0-h-tiny


The models listed should be the same as the ones you subscribed to in the previous step.


## 2.4 Testing the Models’ Existing Knowledge

Ask a common RPG question

Next, we will test the language models’ innate knowledge of our name space. This is what many customers would do. Run the following code in a new cell to define the `test_all_models` function.

Now, run the following code to see what the models know about the Basic Fantasy game


In [8]:
def test_all_models(api_key: str, base_url: str, question: str, max_tokens: int = 1000):

    url_models = f"{base_url}/models"
    url_chat = f"{base_url}/chat/completions"

    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }

    # 1. Get all models
    try:
        response = requests.get(url_models, headers=headers)
        response.raise_for_status()
        models_data = response.json().get("data", [])
    except Exception as e:
        print(f"❌ Failed to fetch models: {e}")
        return

    print(f"\n--- Running prompt against {len(models_data)} models ---\n")

    # 2. Loop through each model
    for model in models_data:
        model_id = model["id"]
        print(f"🤖 Testing Model: {model_id}...")

        payload = {
            "model": model_id,
            "messages": [{"role": "user", "content": question}],
            "max_tokens": max_tokens
        }

        try:
            res = requests.post(url_chat, headers=headers, json=payload)
            res.raise_for_status()

            answer = res.json()["choices"][0]["message"]["content"]
            print(f"✅ Response:\n{answer}\n")

        except Exception as e:
            print(f"❌ Failed to get response from {model_id}: {e}\n")

    print("--- All tests complete ---")

Now, run the following code to see what the models know about the Basic Fantasy game


In [9]:
test_all_models(
    api_key=key,
    base_url=endpoint_base,
    question="What happens if a Thief fails an Open Locks attempt?"
)


--- Running prompt against 4 models ---

🤖 Testing Model: microsoft-phi-4...
✅ Response:
In many role-playing games (RPGs), especially those like Dungeons & Dragons (D&D), a Thief—or a similar class focused on thievery, skills, and dexterity—might encounter situations where they need to attempt to open locks. The specific consequences of failing an Open Locks attempt can vary based on the game system and the Dungeon Master's (DM's) or Game Master's (GM's) rules. Here are some general possibilities:

1. **No Immediate Consequences**: In some games or scenarios, failing to open a lock simply means the character cannot proceed with that specific action. They may need to wait until they are able to attempt it again, perhaps with the aid of tools or another character's skills.

2. **Damaged Locks**: If a lock is tampered with and the attempt to open it fails, the lock may become more complex to open in subsequent attempts. This often occurs in scenarios where maintaining stealth or avoidin

You'll notice that the output will vary slightly across each model, but the LLM will hedge their bets by saying something to the effect of "In many fantasy role-playing games, including Dungeons & Dragons,"  The customer's game is not Dungeons & Dragons and we want specific to this game.

Given that the models' innate answer mentions the customer's top competitor is certainly a sore point. 

Once again, the models offer results that generalize based on whatever public documents it had access to during training. 

For reference, on page 9 of the rule book, the correct answer is below: 

>Open Locks allows the Thief to unlock a lock without a proper key. It may only be tried once per lock. If the attempt fails, the Thief must wait until they have gained another level of experience before trying again.

## 2.5 The Customization Hierarchy

Before we jump to solutions, it helps to name the full spectrum of options for connecting a model to customer data. There are three main levers, each building on the last:

| Lever | What It Controls | When It Helps |
|-------|-----------------|---------------|
| **Prompt Engineering** | *How* the model reasons and responds | Improving format, tone, and structure of answers |
| **Retrieval-Augmented Generation (RAG)** | *What* the model knows at inference time | Grounding answers in specific documents |
| **Fine-tuning** | *What* the model has internalized | Teaching domain-specific patterns, terminology, or behavior |

No matter which approach you use, prompting underpins them all. It defines how the model behaves at inference time. A RAG pipeline with a poor prompt will produce poor answers even with perfect retrieval. Fine-tuning without deliberate prompting will underperform.

So why not stop at better prompts?

Because in our case the model is not *reasoning poorly*. It is missing the information entirely. No prompt can force a model to produce the Basic Fantasy RPG rule that a failed Open Locks attempt locks the Thief out until the next experience level. That fact was never in the training data.

Prompt engineering is the foundation. RAG adds the knowledge. Fine-tuning, if needed, comes last. We will follow that order for the rest of this workshop.

**Next:** We ingest the customer's documents and make them available to the model.